# Framing a vague business problem into a clear ML task

_Doing ML for Real — the skills that matter (2026)_

**Turn 'reduce churn' into a precise spec: a decision, a target, a label, a metric, and proof there is signal.**

A business ask is a wish. An ML task is a contract. Framing is the work of turning one into the other.

       The contract says exactly four things: what you predict (the label), for what unit (the row), to drive what decision (the action), and measured how (the metric and baseline).

---

This notebook is a practice scaffold for the **Framing a vague business problem into a clear ML task** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scikit-learn + pandas

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

# ----- load the framed dataset -----
# one row per UNIT OF ANALYSIS (e.g. one user); y is the LABEL you defined
# with a horizon, e.g. 1 = "no login for 30 days in the next 60", 0 = active.
df = pd.read_parquet("churn_features.parquet")
y = df.pop("churned").to_numpy()          # the positive class is "churned"
X = df.to_numpy()
feature_names = list(df.columns)

# ----- (1) base rate: how often is the positive class true? -----
base_rate = y.mean()
majority_baseline = max(base_rate, 1 - base_rate)   # accuracy of "always predict the majority"
print(f"base rate (P churn) = {base_rate:.3f}")
print(f"majority-class baseline accuracy = {majority_baseline:.3f}")

# ----- (2) is there SIGNAL? mutual information per candidate feature -----
mi = mutual_info_classif(X, y, random_state=0)      # >0 => the feature carries info about y
ranking = sorted(zip(feature_names, mi), key=lambda t: t[1], reverse=True)
print("\ntop candidate features by mutual information:")
for name, score in ranking[:8]:
    print(f"  {name:<22} I(X;y) = {score:.3f}")

# ----- (3) does a 5-line baseline BEAT chance? -----
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
acc = cross_val_score(clf, X, y, cv=5).mean()        # honest out-of-fold accuracy
print(f"\nlogistic-regression CV accuracy = {acc:.3f}")
print("VERDICT:", "signal beats chance -> frame is feasible"
      if acc > majority_baseline + 0.02 else
      "no lift over baseline -> rethink the frame or kill it")

## Visualize the data & results

_Before building a churn-style model, is there any signal to frame a task around? Which candidate features actually carry information about the label?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

data = load_breast_cancer()           # 569 real tumor records, 30 features
X, y = data.data, data.target         # label: 1 = benign, 0 = malignant

# base rate + majority baseline (what any model must beat)
base_rate = y.mean()                                  # 0.627 (P benign)
majority = max(base_rate, 1 - base_rate)              # 0.627
print("base rate P(benign) =", round(float(base_rate), 3))
print("majority baseline   =", round(float(majority), 3))

# mutual information: which features carry signal about the label?
mi = mutual_info_classif(X, y, random_state=0)
order = np.argsort(mi)[::-1][:7]                       # top 7 candidates
for i in order:
    print(f"{data.feature_names[i]:<22} {mi[i]:.3f}")
# top 7 -> [0.473, 0.463, 0.454, 0.439, 0.437, 0.403, 0.375]

# does a 5-line baseline beat chance?
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
print("logreg CV accuracy =", round(float(cross_val_score(clf, X, y, cv=5).mean()), 3))
# -> 0.981, well above the 0.627 majority baseline: signal is real

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A product manager says "use ML to reduce customer churn". Turn this into a one-line ML task: name the decision, the unit, the framing, and the label with a horizon.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Find the decision and actor. — _Ask what action changes. Here: the retention team sends a discount to at-risk users. No action means no task._
- Set the unit of analysis to per-user (the actor acts on a user). — _Each scored row is one user, because the offer goes to a user, not a session or a transaction._
- Pick the framing from the action shape: a fixed offer budget means ranking, not raw classification. — _With limited budget you want the highest-risk users first, which is a ranking, scored by churn probability._
- Define the label with a clock: "no login for 30 consecutive days within the next 60 days". — _A measurable event plus a horizon makes the label computable from logs and aligned with when the offer can still help._

**Answer:** Per-user, rank users by P(no login for 30 consecutive days within the next 60), so the retention team's fixed-budget discount goes to the highest-risk users. (Ranking, not classification, because the budget is fixed.)

</details>

**Problem 2.** Before modeling, how do you cheaply check there is any signal to predict churn, and why is "95% accuracy" not enough to declare success?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute the base rate of the positive class. — _If only 5% of users churn, predicting "stays" for everyone is already 95% accurate, so accuracy alone is meaningless._
- Run mutual_info_classif on candidate features against the label. — _Mutual information measures any dependence between a feature and the label; near-zero for all features means no model can beat the base rate._
- Fit a 5-line LogisticRegression with cross-validation and compare to the majority-class baseline. — _If the cross-validated metric does not beat the trivial baseline, the project is infeasible and should be killed now, cheaply._

**Answer:** Estimate the base rate, run mutual_info_classif on candidate features to confirm non-zero signal, then check a 5-line cross-validated LogisticRegression beats the majority baseline. "95% accuracy" is worthless if 95% of users don't churn — you must beat the base rate, not match it.

</details>